# CFPB Consumer Complaint NLP Analysis
**What Do 383,000 Financial Complaints Reveal?**

Analyzed 383,564 CFPB complaint narratives to surface systemic patterns, predict resolution outcomes, and identify which companies drive the most regulatory scrutiny.

**Three research questions:**
- Q1: What topics dominate consumer complaints?
- Q2: Can complaint text predict whether a company responds on time?
- Q3: Do complaint narratives differ by product in sentiment and length?

## 1. Setup & Imports

In [ ]:
# Install dependencies
!pip install wordcloud gensim transformers sentencepiece spacy imbalanced-learn -q
!python -m spacy download en_core_web_sm -q

import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from collections import Counter
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# Portfolio color scheme
# ------------------------------------------------------------
BG     = '#0a0a0a'
LIME   = '#c8ff00'
PINK   = '#ff69b4'
WHITE  = '#ffffff'
GRAY   = '#333333'
MGRAY  = '#1a1a1a'

plt.rcParams.update({
    'figure.facecolor':  BG,
    'axes.facecolor':    BG,
    'axes.edgecolor':    GRAY,
    'axes.labelcolor':   WHITE,
    'xtick.color':       WHITE,
    'ytick.color':       WHITE,
    'text.color':        WHITE,
    'grid.color':        MGRAY,
    'axes.titleweight':  'bold',
    'axes.titlesize':    13,
    'axes.labelsize':    11,
    'figure.figsize':    (11, 6),
})

PLOTS_DIR = '/content/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

def apply_style(ax):
    ax.set_facecolor(BG)
    ax.grid(True, axis='y', color=MGRAY, linewidth=0.8)
    ax.grid(False, axis='x')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.spines['left'].set_color(GRAY)
    ax.spines['bottom'].set_color(GRAY)

def save_plot(filename):
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, filename), dpi=300,
                bbox_inches='tight', facecolor=BG)
    plt.show()
    plt.close()

print('Setup complete.')

## 2. Load Dataset

> Upload CFPB.csv to /content/ before running this cell.

In [ ]:
cols_needed = [
    'Consumer complaint narrative',
    'Product', 'Sub-product',
    'Issue', 'Sub-issue',
    'Timely response?',
    'State', 'Submitted via',
    'Date received', 'Company'
]

df = pd.read_csv('/content/CFPB.csv', usecols=cols_needed, low_memory=False)
df = df[df['Consumer complaint narrative'].notna()].copy()

df = df.rename(columns={
    'Consumer complaint narrative': 'narrative',
    'Product': 'product',
    'Sub-product': 'sub_product',
    'Issue': 'issue',
    'Sub-issue': 'sub_issue',
    'Timely response?': 'timely_response',
    'State': 'state',
    'Submitted via': 'channel',
    'Date received': 'date',
    'Company': 'company'
})

df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['narrative_length'] = df['narrative'].apply(lambda x: len(str(x).split()))

print('=' * 60)
print('CORPUS SUMMARY')
print('=' * 60)
print(f'Total narratives:   {len(df):,}')
print(f'Date range:         {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Unique products:    {df["product"].nunique()}')
print(f'Unique companies:   {df["company"].nunique():,}')
print(f'Unique states:      {df["state"].nunique()}')
print(f'\nTimely response breakdown:')
print(df['timely_response'].value_counts())
print(f'\nNarrative length:')
print(df['narrative_length'].describe().round(1))

df.to_csv('/content/cfpb_clean.csv', index=False)
print('\nDataset saved.')

## 3. Text Preprocessing

In [ ]:
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
stop_words.update([
    'x', 'xx', 'xxx', 'xxxx', 'xxxxxxxxxxxx', 'company',
    'account', 'bank', 'would', 'also', 'one', 'get',
    'said', 'told', 'called', 'made', 'know', 'back'
])

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\b[xX]+\b', '', text)   # remove XXXX redactions
    text = re.sub(r'[^a-z\s]', '', text)     # keep only letters
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

print('Cleaning narratives — takes a few minutes...')
df['clean_narrative'] = df['narrative'].apply(clean_text)
print('Done.')
print(df['clean_narrative'].head(2))

## 4. Q1 — Topic & Language Analysis

**What are people actually complaining about?**

In [ ]:
# --- 4A. Word Frequency & Zipf's Law ---
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

all_words = ' '.join(df['clean_narrative']).split()
word_freq = Counter(all_words)
top_words = word_freq.most_common(20)
words, counts = zip(*top_words)

fig, ax = plt.subplots()
ax.bar(words, counts, color=LIME)
ax.set_title('Top 20 Most Frequent Words in Complaint Narratives')
ax.set_xlabel('Word')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
apply_style(ax)
save_plot('word_freq.png')

# Zipf's Law
sorted_counts = sorted(word_freq.values(), reverse=True)
ranks = range(1, len(sorted_counts) + 1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(list(ranks)[:10000], sorted_counts[:10000], color=PINK, linewidth=1.5)
ax.set_title("Zipf's Law — Word Frequency vs Rank")
ax.set_xlabel('Rank (log)')
ax.set_ylabel('Frequency (log)')
apply_style(ax)
save_plot('zipf.png')

print(f"Zipf validation: top 10 words = {sum(counts[:10]) / sum(word_freq.values()):.1%} of all tokens")

In [ ]:
# --- 4B. Word Cloud ---
from wordcloud import WordCloud

wc_text = ' '.join(df['clean_narrative'].sample(10000, random_state=42))
wordcloud = WordCloud(
    width=1400, height=700,
    background_color='black',
    max_words=150,
    color_func=lambda *args, **kwargs: (
        '#c8ff00' if np.random.random() > 0.5 else '#ff69b4'
    )
).generate(wc_text)

fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor(BG)
ax.imshow(wordcloud, interpolation='bilinear')
ax.axis('off')
ax.set_title('Complaint Vocabulary Across 383,564 Narratives', color=WHITE, pad=12)
save_plot('wordcloud_all.png')

In [ ]:
# --- 4C. TF-IDF by Product ---
top_products = df['product'].value_counts().head(3).index

print('=' * 60)
print('TOP TF-IDF TERMS BY PRODUCT CATEGORY')
print('=' * 60)
for prod in top_products:
    subset = df[df['product'] == prod]['clean_narrative']
    tfidf = TfidfVectorizer(max_features=10, ngram_range=(1, 2))
    tfidf.fit(subset)
    print(f"\n{prod}:")
    print('  ' + ', '.join(tfidf.get_feature_names_out()))

In [ ]:
# --- 4D. N-grams ---
def get_top_ngrams(corpus, n, top_k=15):
    vec = CountVectorizer(ngram_range=(n, n), max_features=10000).fit(corpus)
    bag = vec.transform(corpus)
    sum_words = bag.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    return sorted(words_freq, key=lambda x: x[1], reverse=True)[:top_k]

sample_corpus = df['clean_narrative'].sample(50000, random_state=42)
bigrams  = get_top_ngrams(sample_corpus, 2)
trigrams = get_top_ngrams(sample_corpus, 3)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, ngrams, title, color in zip(
    axes, [bigrams, trigrams],
    ['Top 15 Bigrams', 'Top 15 Trigrams'],
    [LIME, PINK]
):
    labels, vals = zip(*ngrams)
    ax.barh(labels, vals, color=color)
    ax.set_title(title)
    ax.invert_yaxis()
    apply_style(ax)
save_plot('ngrams.png')

In [ ]:
# --- 4E. LDA Topic Modeling ---
from sklearn.decomposition import LatentDirichletAllocation

TOPIC_LABELS = [
    'Mortgage & Foreclosure',
    'Consumer Protection Law',
    'Debt & Collections',
    'Credit Card Disputes',
    'Student Loans',
    'Payment & Billing',
    'Credit Report Errors',
    'Phone Harassment'
]

print('Running LDA topic modeling on 30K sample...')
count_vec = CountVectorizer(max_features=2000, max_df=0.9, min_df=10)
lda_sample = df['clean_narrative'].sample(30000, random_state=42)
lda_matrix = count_vec.fit_transform(lda_sample)
lda_features = count_vec.get_feature_names_out()

lda = LatentDirichletAllocation(n_components=8, random_state=42, max_iter=10, n_jobs=-1)
lda.fit(lda_matrix)

print('\n=== LDA TOPICS (Q1) ===')
for i, (topic, label) in enumerate(zip(lda.components_, TOPIC_LABELS)):
    top_words = [lda_features[j] for j in topic.argsort()[-10:]]
    print(f"Topic {i+1} | {label}: {', '.join(reversed(top_words))}")

lda_output = lda.transform(lda_matrix)
dominant_topics = lda_output.argmax(axis=1)

# Topic distribution chart
topic_counts = pd.Series(dominant_topics).value_counts().sort_index()
topic_counts.index = [TOPIC_LABELS[i] for i in topic_counts.index]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(range(len(topic_counts)), topic_counts.values, color=LIME)
ax.set_xticks(range(len(topic_counts)))
ax.set_xticklabels(topic_counts.index, rotation=35, ha='right', fontsize=9)
ax.set_title('Complaint Distribution Across 8 LDA Topics')
ax.set_ylabel('Count')
apply_style(ax)
save_plot('lda_topics.png')

## 5. Q3 — Sentiment & Narrative Variation

**Do complaint narratives differ by product type in sentiment, length, or topic distribution?**

In [ ]:
import nltk
nltk.download('vader_lexicon', quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

print('Running sentiment analysis on 20K sample...')
sample_df = df.sample(20000, random_state=42).copy()
sample_df['sentiment'] = sample_df['narrative'].apply(
    lambda x: sia.polarity_scores(str(x))['compound']
)
sample_df['sentiment_label'] = sample_df['sentiment'].apply(
    lambda x: 'positive' if x > 0.05 else ('negative' if x < -0.05 else 'neutral')
)

print('Sentiment distribution:')
print(sample_df['sentiment_label'].value_counts())

In [ ]:
# Sentiment distribution
fig, ax = plt.subplots(figsize=(8, 5))
vals = sample_df['sentiment_label'].value_counts()
ax.bar(vals.index, vals.values, color=[PINK, LIME, GRAY])
ax.set_title('Sentiment Distribution Across All Complaints')
ax.set_xlabel('Sentiment')
ax.set_ylabel('Count')
apply_style(ax)
save_plot('sentiment_dist.png')

# Sentiment by product
fig, ax = plt.subplots(figsize=(14, 7))
prod_sentiment = sample_df.groupby('product')['sentiment'].mean().sort_values()
colors = [LIME if v >= 0 else PINK for v in prod_sentiment.values]
ax.barh(prod_sentiment.index, prod_sentiment.values, color=colors)
ax.axvline(0, color=WHITE, linewidth=0.8, linestyle='--')
ax.set_title('Average Sentiment Score by Product Category (Q3)')
ax.set_xlabel('Mean VADER Compound Score')
apply_style(ax)
save_plot('sentiment_by_product.png')

# Sentiment by timely response (Q2 preview)
fig, ax = plt.subplots(figsize=(7, 5))
resp_sentiment = sample_df.groupby('timely_response')['sentiment'].mean()
ax.bar(resp_sentiment.index, resp_sentiment.values, color=[PINK, LIME])
ax.set_title('Average Sentiment by Timely Response (Q2 Preview)')
ax.set_ylabel('Mean VADER Compound Score')
apply_style(ax)
save_plot('sentiment_by_response.png')

# Narrative length by product
fig, ax = plt.subplots(figsize=(14, 7))
df.groupby('product')['narrative_length'].mean().sort_values().plot(
    kind='barh', color=LIME, ax=ax
)
ax.set_title('Average Narrative Length by Product (Q3)')
ax.set_xlabel('Average Word Count')
apply_style(ax)
save_plot('narrative_length_by_product.png')

## 6. Q3 — Linguistic Feature Extraction (spaCy NER & POS)

Extracting named entities and part-of-speech distributions to understand who and what is mentioned most.

In [ ]:
import spacy

nlp = spacy.load('en_core_web_sm', disable=['parser'])

sample = df['narrative'].dropna().sample(2000, random_state=42).tolist()

pos_counts    = Counter()
entity_types  = Counter()
org_entities  = Counter()

print('Running NER and POS tagging... takes a few minutes')
for doc in nlp.pipe(sample, batch_size=50):
    for token in doc:
        if not token.is_stop and not token.is_punct:
            pos_counts[token.pos_] += 1
    for ent in doc.ents:
        entity_types[ent.label_] += 1
        if ent.label_ == 'ORG':
            org_entities[ent.text.lower()] += 1

# Clean org entities — remove redaction noise
noise = {'xxxx', 'xx', 'xxx', 'cfpb', 'fcra', 'fha', 'n/a', 'us', 'u.s'}
clean_orgs = {k: v for k, v in org_entities.items()
              if k not in noise and 'xxxx' not in k and len(k) > 2}

print('NER and POS tagging complete.')

In [ ]:
# POS distribution
fig, ax = plt.subplots(figsize=(10, 5))
labels, vals = zip(*pos_counts.most_common(10))
ax.bar(labels, vals, color=LIME)
ax.set_title('POS Tag Distribution in Complaint Narratives')
ax.set_xlabel('POS Tag')
ax.set_ylabel('Count')
apply_style(ax)
save_plot('pos_tags.png')

# Entity types
fig, ax = plt.subplots(figsize=(10, 5))
ent_labels, ent_vals = zip(*entity_types.most_common(10))
ax.bar(ent_labels, ent_vals, color=PINK)
ax.set_title('Named Entity Types in Complaint Narratives')
apply_style(ax)
save_plot('entity_types.png')

# Top organizations
top_orgs = dict(sorted(clean_orgs.items(), key=lambda x: x[1], reverse=True)[:15])
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(list(top_orgs.keys()), list(top_orgs.values()), color=LIME)
ax.set_title('Top 15 Organizations Mentioned in Complaints')
ax.invert_yaxis()
apply_style(ax)
save_plot('top_orgs.png')

## 7. Q2 — Predictive Modeling: Timely Response

**Can complaint text predict whether a company responds on time?**

Binary classification task with severe class imbalance: 97% timely vs 3% untimely.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

model_df = df[['clean_narrative', 'timely_response']].dropna().copy()
model_df = model_df[model_df['timely_response'].isin(['Yes', 'No'])]
X = model_df['clean_narrative']
y = (model_df['timely_response'] == 'Yes').astype(int)

print(f'Class distribution:\n{y.value_counts()}')
print(f'\nImbalance ratio: {y.value_counts()[1] / y.value_counts()[0]:.0f}:1 (timely:untimely)')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Model 1 — Logistic Regression with class_weight balanced
print('\nTraining Logistic Regression (balanced)...')
lr_balanced = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, random_state=42,
                               class_weight='balanced', n_jobs=-1))
])
lr_balanced.fit(X_train, y_train)
lr_preds  = lr_balanced.predict(X_test)
lr_proba  = lr_balanced.predict_proba(X_test)[:, 1]
lr_auc    = roc_auc_score(y_test, lr_proba)
print(classification_report(y_test, lr_preds))
print(f'LR ROC-AUC: {lr_auc:.4f}')

# Model 2 — Naive Bayes with undersampling
print('\nTraining Naive Bayes (undersampled)...')
train_df  = pd.DataFrame({'text': X_train, 'label': y_train})
majority  = train_df[train_df['label'] == 1]
minority  = train_df[train_df['label'] == 0]
maj_down  = resample(majority, replace=False,
                     n_samples=len(minority) * 10, random_state=42)
balanced  = pd.concat([maj_down, minority])

nb_balanced = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf', MultinomialNB())
])
nb_balanced.fit(balanced['text'], balanced['label'])
nb_preds  = nb_balanced.predict(X_test)
nb_proba  = nb_balanced.predict_proba(X_test)[:, 1]
nb_auc    = roc_auc_score(y_test, nb_proba)
print(classification_report(y_test, nb_preds))
print(f'NB ROC-AUC: {nb_auc:.4f}')

print('\n=== MODEL COMPARISON ===')
print(f'LR Balanced ROC-AUC:   {lr_auc:.4f}  ← winner')
print(f'NB Undersampled ROC-AUC: {nb_auc:.4f}')

In [ ]:
# ROC Curve — portfolio style
fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay.from_predictions(
    y_test, lr_proba,
    name=f'Logistic Regression (balanced) · AUC = {lr_auc:.2f}',
    ax=ax, color=LIME
)
RocCurveDisplay.from_predictions(
    y_test, nb_proba,
    name=f'Naive Bayes (undersampled) · AUC = {nb_auc:.2f}',
    ax=ax, color=PINK
)
ax.plot([0, 1], [0, 1], linestyle='--', color=GRAY, linewidth=1)
ax.set_title('Model Comparison: Timely Response Classifier (Q2)')
ax.set_facecolor(BG)
ax.legend(frameon=False)
apply_style(ax)
save_plot('roc_curve.png')

# Top predictive features
tfidf_fitted  = lr_balanced.named_steps['tfidf']
lr_coefs      = lr_balanced.named_steps['clf'].coef_[0]
feature_names = tfidf_fitted.get_feature_names_out()

top_pos = sorted(zip(feature_names, lr_coefs), key=lambda x: x[1], reverse=True)[:15]
top_neg = sorted(zip(feature_names, lr_coefs), key=lambda x: x[1])[:15]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, data, title, color in zip(
    axes, [top_pos, top_neg],
    ['Words Predicting TIMELY Response', 'Words Predicting UNTIMELY Response'],
    [LIME, PINK]
):
    words_f, coefs = zip(*data)
    ax.barh(words_f, coefs, color=color)
    ax.set_title(title)
    ax.invert_yaxis()
    apply_style(ax)
save_plot('feature_importance.png')

## 8. Advanced NLP

Applying Word2Vec embeddings, zero-shot classification (BART-MNLI), and abstractive summarization (BART-CNN).

In [ ]:
# --- 8A. Word2Vec + t-SNE ---
from gensim.models import Word2Vec
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

print('Training Word2Vec on full corpus...')
sentences = [text.split() for text in df['clean_narrative'].dropna()]
w2v_model = Word2Vec(
    sentences, vector_size=100, window=5,
    min_count=10, workers=4, seed=42
)
print(f'Vocabulary: {len(w2v_model.wv):,} words')

print('\n=== SEMANTICALLY SIMILAR WORDS ===')
for word in ['credit', 'debt', 'mortgage', 'payment', 'fraud']:
    if word in w2v_model.wv:
        similar = w2v_model.wv.most_similar(word, topn=5)
        print(f"'{word}': {[w for w, _ in similar]}")

In [ ]:
# t-SNE visualization
target_words = {
    'Credit Reporting': ['accounts', 'credit', 'equifa', 'information', 'inquiry', 'report', 'reporting'],
    'Debt Collection':  ['call', 'collection', 'debt', 'letter', 'never', 'phone', 'received'],
    'Mortgage':         ['home', 'loan', 'modification', 'mortgage', 'payment', 'property'],
}

filtered  = {cat: [w for w in words if w in w2v_model.wv] for cat, words in target_words.items()}
all_w     = [(word, cat) for cat, words in filtered.items() for word in words]
word_list = [w for w, _ in all_w]
cats      = [c for _, c in all_w]
vectors   = np.array([w2v_model.wv[w] for w in word_list])

pca_r    = PCA(n_components=min(20, len(vectors) - 1), random_state=42)
vec_pca  = pca_r.fit_transform(vectors)
tsne     = TSNE(n_components=2, random_state=42, perplexity=5, n_iter=1000)
vec_2d   = tsne.fit_transform(vec_pca)

CAT_COLORS = {'Credit Reporting': LIME, 'Debt Collection': PINK, 'Mortgage': WHITE}

fig, ax = plt.subplots(figsize=(9, 6))
for cat in target_words:
    mask = [c == cat for c in cats]
    x = vec_2d[mask, 0]
    y = vec_2d[mask, 1]
    ax.scatter(x, y, label=cat, color=CAT_COLORS[cat], s=80, alpha=0.9)
    for i, word in enumerate([w for w, c in all_w if c == cat]):
        ax.annotate(word, (x[i], y[i]), fontsize=8, color=WHITE,
                    xytext=(5, 5), textcoords='offset points')
ax.set_title('Word2Vec Embeddings — t-SNE Visualization by Complaint Category')
ax.legend(frameon=False)
apply_style(ax)
save_plot('word2vec_tsne.png')

# Cosine similarity heatmap
centroids = {cat: np.mean([w2v_model.wv[w] for w in words], axis=0)
             for cat, words in filtered.items() if words}
cat_names = list(centroids.keys())
sim_matrix = cosine_similarity(np.array([centroids[c] for c in cat_names]))
sim_df = pd.DataFrame(sim_matrix, index=cat_names, columns=cat_names)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(sim_df, annot=True, fmt='.2f',
            cmap='YlOrRd', linewidths=0.5, vmin=0, vmax=1, ax=ax)
ax.set_title('Semantic Similarity Between Complaint Categories (Word2Vec)')
plt.tight_layout()
save_plot('category_similarity.png')

In [ ]:
# --- 8B. Zero-Shot Classification (BART-MNLI) ---
from transformers import pipeline

print('Loading zero-shot model (facebook/bart-large-mnli)...')
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

candidate_labels = [
    'credit reporting error',
    'debt collection harassment',
    'billing dispute',
    'loan modification issue',
    'fraud or identity theft',
    'account closure',
    'customer service failure',
    'mortgage foreclosure'
]

sample_texts = df['narrative'].dropna().sample(10, random_state=42).tolist()

print('\n=== ZERO-SHOT CLASSIFICATION RESULTS ===')
results = []
for i, text in enumerate(sample_texts):
    truncated = ' '.join(text.split()[:200])
    result    = classifier(truncated, candidate_labels)
    top_label = result['labels'][0]
    top_score = result['scores'][0]
    print(f'Complaint {i+1}: {top_label} (score: {top_score:.2f})')
    results.append({'complaint': i + 1, 'label': top_label, 'score': top_score})

results_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(10, 5))
results_df['label'].value_counts().plot(kind='barh', color=LIME, ax=ax)
ax.set_title('Zero-Shot Classification — Complaint Issue Types')
ax.set_xlabel('Count')
apply_style(ax)
save_plot('zeroshot.png')

In [ ]:
# --- 8C. Abstractive Summarization (BART-CNN) ---
from transformers import BartForConditionalGeneration, BartTokenizer

print('Loading summarization model (facebook/bart-large-cnn)...')
model_name = 'facebook/bart-large-cnn'
tokenizer  = BartTokenizer.from_pretrained(model_name)
bart_model = BartForConditionalGeneration.from_pretrained(model_name)

def summarize(text, max_length=80, min_length=30):
    inputs = tokenizer(text, return_tensors='pt', max_length=1024, truncation=True)
    summary_ids = bart_model.generate(
        inputs['input_ids'], max_length=max_length,
        min_length=min_length, length_penalty=2.0,
        num_beams=4, early_stopping=True
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

categories = {
    'Credit Reporting': df[df['product'].str.contains('Credit reporting', case=False, na=False)],
    'Debt Collection':  df[df['product'] == 'Debt collection'],
    'Mortgage':         df[df['product'] == 'Mortgage'],
}

print('\n=== ABSTRACTIVE SUMMARIZATION BY COMPLAINT CATEGORY ===')
for cat, cat_df in categories.items():
    sample_texts  = cat_df['narrative'].dropna().sample(20, random_state=42).tolist()
    combined      = ' '.join(' '.join(t.split()[:45]) for t in sample_texts)
    combined      = ' '.join(combined.split()[:900])
    summary       = summarize(combined)
    print(f'\nCategory: {cat}')
    print(f'Summary:  {summary}')
    print('-' * 60)

print('\nNote: XXXX masking degrades summarization quality — a real-world data limitation.')

## 9. Key Findings Summary

In [ ]:
print('=' * 70)
print('KEY FINDINGS SUMMARY')
print('=' * 70)

print("""
Q1 — TOPIC & LANGUAGE ANALYSIS
  - LDA identified 8 distinct complaint themes mapping cleanly onto
    product categories. Payment & Billing dominated the corpus.
  - TF-IDF confirms distinct vocabulary per product — enabling
    automated complaint routing.
  - Zipf's Law confirmed: top 10 words account for 40%+ of all tokens.

Q2 — PREDICTIVE MODELING
  - LR (ROC-AUC: 0.79) outperforms NB (0.78).
  - LR captures 66% of untimely cases vs 10% for NB — the metric
    that matters for flagging at-risk complaints.
  - Predictive signal comes from which company is named, not
    emotional tone of the complaint.

Q3 — NARRATIVE VARIATION
  - Sentiment is predominantly negative across all products.
  - Mortgage complaints show the most distinct linguistic profile.
  - Zero-shot classification: customer service failure dominates —
    most complaints are process breakdowns, not product failures.

ADVANCED NLP
  - Word2Vec trained on 383K narratives (23K+ word vocabulary).
  - Credit Reporting and Debt Collection share moderate semantic
    similarity (0.38); Mortgage is semantically distinct.
  - XXXX masking degrades summarization — a real-world data
    quality challenge surfaced by this analysis.
""")

## 10. Preview All Charts

In [ ]:
from IPython.display import Image, display

charts = [
    'wordcloud_all.png',
    'lda_topics.png',
    'roc_curve.png',
    'feature_importance.png',
    'sentiment_by_product.png',
    'top_orgs.png',
    'word2vec_tsne.png',
    'zeroshot.png',
    'ngrams.png',
    'narrative_length_by_product.png',
]

for chart in charts:
    path = os.path.join(PLOTS_DIR, chart)
    if os.path.exists(path):
        print(f'--- {chart} ---')
        display(Image(filename=path))